In [2]:
cd ..

/home/karaman/Documents/bet


In [3]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity



In [ ]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/24 21:54:47 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.182.129 instead (on interface ens33)
26/01/24 21:54:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/24 21:54:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')


In [6]:
first_last_activity = generate_last_activity(silver_money_events)
first_last_activity = (
    first_last_activity
    .join(
        players_silver.select("player_idx"),
        on="player_idx",
        how="inner"
    )
)


In [7]:
player_snapshot = (players_silver
                   .select('player_idx','country','age_bucket','device_type',
                           'acquisition_channel', 'registration_date', 'risk_segment', 
                           'lifecycle_stage', F.col('current_balance'))
                   .join(first_last_activity,
                         on='player_idx',
                         how='left')
)

## Construct training dataset for ML

### Full dataframe (players x dates)

In [8]:
date_range = spark.sql(
    f"SELECT explode(sequence(to_date('{config_.start_date}'), to_date('{config_.end_date}'), interval 1 day)) AS reference_date"
)

df_pl_date = (player_snapshot
              .select('player_idx', 'registration_date','first_event_date', 'last_event_date',  'first_session_date',  'last_session_date', 'first_financial_date', 'last_financial_date')
              .crossJoin(date_range)
            .filter(F.col('first_session_date').isNotNull())  # exclude new players
            .withColumn('session_date_days', 
                F.datediff(F.col("reference_date"), F.lit("1970-01-01")))
)
window_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-6,0))
window_30d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-29,0))

In [9]:
df_pl_date.count()

11980878

### Compute rolling futures based on sessions

In [10]:
df_sessions_detail = (df_pl_date.select('player_idx','reference_date','registration_date', 'first_session_date','last_session_date','session_date_days')
.join(sessions_silver
      .select('session_id',
 'session_duration_sec',
 'bet_count',
 'total_bet_amount',
 'total_win_amount',
 'signed_amount',
 'balance_after_txn',
   F.col('session_date').alias('reference_date'), 
   'player_idx'),
        how='left', on=['player_idx','reference_date'])
.filter( F.col("first_session_date")<= F.col("reference_date"))
#.filter(F.datediff(F.col("reference_date"), F.col("first_session_date")) > 30)
#.filter(F.datediff(F.lit(config_.end_date), F.col("reference_date")) > 7)
.orderBy('player_idx', 'reference_date' )
)

In [11]:
df_sessions_rolling= (df_sessions_detail
            .withColumn('num_sessions_7d', F.count('session_id').over(window_7d))
            .withColumn('net_game_result_7d',F.coalesce(F.sum('signed_amount').over(window_7d), F.lit(0)))
            .withColumn('num_sessions_30d', F.count('session_id').over(window_30d))
            .withColumn('net_game_result_30d', F.coalesce(F.sum('signed_amount').over(window_30d), F.lit(0))) 
            .withColumn('avg_sessions_duration_30d', F.coalesce(F.avg('session_duration_sec').over(window_30d), F.lit(0)))
            .withColumn('avg_bet_amount_30d', F.coalesce(F.avg('total_bet_amount').over(window_30d), F.lit(0)))
                       ) 
df_sessions_rolling = df_sessions_rolling.select(
                            'player_idx',
                            'reference_date',
                            'num_sessions_7d',
                            'net_game_result_7d',
                            'num_sessions_30d',
                            'avg_sessions_duration_30d',
                            'avg_bet_amount_30d',
                            'net_game_result_30d',
                            'first_session_date').drop_duplicates()

In [ ]:
df_sessions_rolling.persist()
df_sessions_rolling.count() 

for c in df_sessions_rolling.columns:
   assert df_sessions_rolling.filter(F.col(c).isNull()).count() == 0


DataFrame[player_idx: bigint, reference_date: date, num_sessions_7d: bigint, net_game_result_7d: double, num_sessions_30d: bigint, avg_sessions_duration_30d: double, avg_bet_amount_30d: double, net_game_result_30d: double, first_session_date: date]

### Compute rolling futures based on all events

In [13]:
silver_money_events_detail =  (df_pl_date.select('player_idx','reference_date','registration_date', 'first_event_date','last_event_date','session_date_days')
.join(silver_money_events
      .select('event_id',
 'event_type',
 'signed_amount',
 'balance_after_txn',
   F.to_date(F.col('event_ts')).alias('reference_date'), 
   'player_idx'),
        how='left', on=['player_idx','reference_date'])
.filter( F.col("first_event_date")<= F.col("reference_date"))
.orderBy('player_idx', 'reference_date' )
)

In [14]:
window_up_to_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(Window.unboundedPreceding,-6))
window_up_to_30d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(Window.unboundedPreceding,-29))

silver_money_events_rolling = (silver_money_events_detail
            .withColumn('balance_7d_ago', F.last('balance_after_txn', ignorenulls=True).over(window_up_to_7d))
            .withColumn('net_amount_result_7d', F.coalesce( F.sum('signed_amount').over(window_7d), F.lit(0)))
            .withColumn('balance_30d_ago', F.last('balance_after_txn', ignorenulls=True).over(window_up_to_7d))
            .withColumn('net_amount_result_30d',F.coalesce( F.sum('signed_amount').over(window_30d), F.lit(0)))
                       )

silver_money_events_rolling = (silver_money_events_rolling
.filter(F.datediff(F.col("reference_date"), F.col("first_event_date")) > 30)
.filter(F.datediff(F.lit(config_.end_date), F.col("reference_date")) > 7)
.select(
                            'player_idx',
                            'reference_date',
                            'balance_7d_ago',
                            'balance_30d_ago',
                            'net_amount_result_7d',
                            'net_amount_result_30d').drop_duplicates())


In [ ]:
silver_money_events_rolling.persist()
silver_money_events_rolling.count() 

for c in df_sessions_rolling.columns:
   assert df_sessions_rolling.filter(F.col(c).isNull()).count() == 0


DataFrame[player_idx: bigint, reference_date: date, balance_7d_ago: double, balance_30d_ago: double, net_amount_result_7d: double, net_amount_result_30d: double]

### Compute rolling futures based on financial transactions 

In [16]:
transactions_detail = (df_pl_date.select('player_idx','reference_date','registration_date', 'first_financial_date','last_financial_date','session_date_days')
.join(transactions_silver
      .select('transaction_id',
 'transaction_type',
 'amount',
 'success_flag',
 'signed_amount',
 'balance_after_txn',
   F.to_date(F.col('transaction_ts')).alias('reference_date'), 
   'player_idx'),
        how='left', on=['player_idx','reference_date'])
.filter( F.col("first_financial_date")<= F.col("reference_date"))
#.filter(F.datediff(F.col("reference_date"), F.col("first_session_date")) > 30)
#.filter(F.datediff(F.lit(config_.end_date), F.col("reference_date")) > 7)
.orderBy('player_idx', 'reference_date' )
)


In [17]:

transactions_rolling = (transactions_detail
                .withColumn('failed_withdrawals_30d', 
                    F.sum(F.when((F.col('success_flag')==False) & (F.col('transaction_type')=='withdrawal'), 1).otherwise(0)).over(window_30d))
                .withColumn('deposit_count_30d', 
                    F.sum(F.when(F.col('transaction_type')=='deposit',1).otherwise(0)).over(window_30d))
                .withColumn('withdrawal_count_30d', 
                    F.sum(F.when(F.col('transaction_type')=='withdrawal',1).otherwise(0)).over(window_30d))
                .withColumn('withdrawal_ratio',
                     F.when(
                        F.col("deposit_count_30d") > 0,
                        F.col("withdrawal_count_30d") / F.col("deposit_count_30d")
                         ).otherwise(F.lit(0.0)))
)

transactions_rolling = transactions_rolling.select(
                            'player_idx',
                            'reference_date',
                            'failed_withdrawals_30d',
                            'deposit_count_30d',
                            'withdrawal_count_30d',
                            'withdrawal_ratio',
).drop_duplicates()

In [ ]:
transactions_rolling.persist()
transactions_rolling.count() 

for c in df_sessions_rolling.columns:
   assert df_sessions_rolling.filter(F.col(c).isNull()).count() == 0


DataFrame[player_idx: bigint, reference_date: date, failed_withdrawals_30d: bigint, deposit_count_30d: bigint, withdrawal_count_30d: bigint, withdrawal_ratio: double]

### Compose the final training dataset

In [24]:
gold_player_behavior = (silver_money_events_rolling
                        .join(df_sessions_rolling,
                        how='left', on=['player_idx', 'reference_date'])
                        .join(transactions_rolling,
 how='left', on=['player_idx', 'reference_date']) 
)

silver_money_events_rolling.unpersist()
df_sessions_rolling.unpersist()
transactions_rolling.unpersist()


DataFrame[player_idx: bigint, reference_date: date, failed_withdrawals_30d: bigint, deposit_count_30d: bigint, withdrawal_count_30d: bigint, withdrawal_ratio: double]

In [ ]:
gold_player_behavior.persist()
gold_player_behavior.count()

26/01/24 22:00:06 WARN CacheManager: Asked to cache already cached data.


5374031

In [30]:

## keep only days that follow the first session date of each player since 
gold_player_behavior = (gold_player_behavior
.filter(F.datediff(F.col("reference_date"), F.col("first_session_date")) > 30)     
.filter(F.datediff(F.lit(config_.end_date), F.col("reference_date")) > 7)
.drop(F.col('first_session_date')) )


## in case of sessions or transactions dates, one  follow the other -> null values 
for c in gold_player_behavior.columns[2:]:  # exclude player_idx and reference_date 
    gold_player_behavior = gold_player_behavior.withColumn(
        c, F.coalesce(F.col(c), F.lit(0))
    )

In [ ]:
for c in gold_player_behavior.columns:
    assert gold_player_behavior.filter(F.col(c).isNull()).count() == 0
gold_player_behavior.count()
gold_player_behavior.unpersist()

### Create gold labels

In [33]:
df_num_of_sessions = (sessions_silver
.groupBy('player_idx', 'session_date')
.agg(F.count('*').alias('num_of_session')))
#.orderBy('player_idx', 'session_date' ))

df_sessions = (df_pl_date
.join(df_num_of_sessions
      .select('num_of_session', F.col('session_date').alias('reference_date'), 'player_idx'),
        how='left', on=['player_idx','reference_date'])
)

sessions_silver_sec = (df_sessions
                       .filter(F.col('reference_date') >= F.col('first_session_date'))
            .withColumn('num_sessions_7d', 
                        F.sum(F.when(F.col("num_of_session") > 0, 1).otherwise(0)).over(window_7d))
            .withColumn('churn_7d', F.when(F.col('num_sessions_7d')==0, True).otherwise(False))
            .withColumn('num_sessions_30d', 
                        F.sum(F.when(F.col('num_of_session') > 0, 1).otherwise(0)).over(window_30d))
           .withColumn('churn_30d', F.when(F.col('num_sessions_30d')==0, True).otherwise(False))
)

In [34]:
window_7d_ahead = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(1,6))
target = (sessions_silver_sec
            .withColumn('next_7d_churn', 
            (F.max(F.col("churn_7d").cast("int")).over(window_7d_ahead) == 1))
)



In [42]:
gold_labels = (target
.select('player_idx', 'reference_date','next_7d_churn')
.filter(F.datediff(F.col("reference_date"), F.col("first_session_date")) > 30)
.filter(F.datediff(F.lit(config_.end_date), F.col("reference_date")) > 7))

In [43]:
gold_labels.count()

5371646

In [44]:
gold_labels.show()

+----------+--------------+-------------+
|player_idx|reference_date|next_7d_churn|
+----------+--------------+-------------+
|        19|    2024-05-11|         true|
|        19|    2024-05-12|         true|
|        19|    2024-05-13|         true|
|        19|    2024-05-14|         true|
|        19|    2024-05-15|         true|
|        19|    2024-05-16|         true|
|        19|    2024-05-17|         true|
|        19|    2024-05-18|         true|
|        19|    2024-05-19|        false|
|        19|    2024-05-20|        false|
|        19|    2024-05-21|        false|
|        19|    2024-05-22|        false|
|        19|    2024-05-23|        false|
|        19|    2024-05-24|        false|
|        19|    2024-05-25|        false|
|        19|    2024-05-26|        false|
|        19|    2024-05-27|        false|
|        19|    2024-05-28|        false|
|        19|    2024-05-29|        false|
|        19|    2024-05-30|        false|
+----------+--------------+-------

In [45]:
assert gold_labels.filter(F.col("next_7d_churn").isNull()).count() == 0


In [ ]:
player_snapshot.write.mode("overwrite").parquet("./data/gold/player_snapshot")
gold_player_behavior.write.mode("overwrite").parquet("./data/gold/player_behavior")
gold_labels.write.mode("overwrite").parquet("./data/gold/labels")


## Player_behavior based on the last day

In [ ]:
silver_money_events = silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
sessions_silver = sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
transactions_silver = transactions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date),F.to_date(F.col('transaction_ts'))))

### player_behavior based on sessions activity

In [ ]:
player_behavior = (sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, 1).otherwise(0)).alias('sessions_7d'),
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_game_result_7d'),

            F.count('*').alias('sesions_30d'),
            F.avg(F.col('session_duration_sec')).cast('int').alias('avg_session_duration_sec_30d'),
            F.avg(F.col('total_bet_amount')).alias('avg_bet_amount_30d'),
            F.sum(F.col('signed_amount')).alias('net_game_result_30d'),
 )
                
)
player_behavior.show()

### player behavior based on any money event (financial + bet)

In [ ]:

### Correct ###
silver_money_events_net = (silver_money_events
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_amount_result_7d'),
            F.sum(F.col('signed_amount')).alias('net_amount_result_30d'),
 )        
)

silver_money_events_net.show()

In [ ]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_30d = (silver_money_events
 .filter(F.col('days_diff') >=30)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_30d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_30d.select('player_idx','balance_after_txn'),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_30d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop( 'current_balance', 'balance_after_txn')
)
player_30d_act.show()

In [ ]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_7d = (silver_money_events
 .filter(F.col('days_diff') >=7)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_7d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_7d.select('player_idx',F.col('balance_after_txn')),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_7d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop('current_balance', 'balance_after_txn')
)
player_7d_act.show()

In [ ]:
transaction_behavior = (transactions_silver
.filter(F.col('days_diff')<=30)
.groupBy(F.col('player_idx'))
.agg(
    F.sum(F.when((F.col('transaction_type')=='withdrawal') & (F.col('success_flag')==False),1).otherwise(0)).alias('failed_withdrawals_30d'),
    F.sum(F.when(F.col('transaction_type')=='deposit',1).otherwise(0)).alias('deposit_count_30d'),
    F.sum(F.when(F.col('transaction_type')=='withdrawal',1).otherwise(0)).alias('withdrawal_count_30d'),
)
.withColumn(        'withdrawal_ratio',
    F.when(
            F.col("deposit_count_30d") > 0,
            F.col("withdrawal_count_30d") / F.col("deposit_count_30d")
        ).otherwise(F.lit(0.0))
)
)
transaction_behavior.show()

In [ ]:
gold_player_behavior = (players_silver.select('player_idx')
                        .join(silver_money_events_net, how='left', on='player_idx') 
                        .join(transaction_behavior, how='left', on='player_idx') 
                        .join(player_behavior, how='left', on='player_idx') 
                        .join(player_7d_act, how='left', on='player_idx') 
                        .join(player_30d_act, how='left', on='player_idx') 

)
gold_player_behavior.show()

In [ ]:
gold_player_behavior.columns

In [ ]:
sessions_silver.filter(F.col('player_idx')==2).orderBy('session_date').show()

# Examples 

In [ ]:
sessions_silver_sec.filter(F.col('churn_7d')==True).filter(F.col('reference_date')>'2024-01-07') .orderBy('reference_date').show()

In [ ]:
target.filter(F.col('churn_7d')).filter(F.col('next_7d_churn')==False).show()

In [ ]:
df_sessions.filter(F.col('player_idx')==10059).show()
